In [1]:
#FILE FOR SARAH ON THE 10th OF JUNE
import os
os.chdir('/Users/noeschaerer')

import numpy as np
import time
parameters = open('odw_2023_main/data_files/main/params_0.txt', mode='r')

from pycbc import frame
from pycbc import types

parameters = open('odw_2023_main/data_files/main/params_0.txt', mode='r')

types_parameters = ['#','starting_time', 'time@Maximum', 'redshifted_mass1', 'redshifted_mass2', 'spin1', 'spin1x', 'spin1y', 'spin1z', 'spin2', 'spin2x', 'spin2y', 'spin2z', 'lambda1', 'lambda2', 'z', 'dist', 'ra', 'declination', 'polarization', 'inclination', 'initial phase',  'snr_optimal', 'type_merger']
stop = True

parameters_conv = []
while stop == True:
    #processing the data
    parameters_i = parameters.readline()
    if parameters_i == str(""):
        stop = False
    else :
        parameters_i = parameters_i.split()
        for p in range(len(parameters_i)):
            #changing from str to float
            if p == 0 or p == (len(parameters_i)-1):
                parameters_i[p]= int(parameters_i[p]) 
            else :
                parameters_i[p]= float(parameters_i[p])

        #creating a dict for each set of parameters
        parameters_i_dict = {}
        for p in range(len(parameters_i)):
            parameters_i_dict[types_parameters[p]] = parameters_i[p]
    parameters_conv.append(parameters_i_dict)
print(parameters_conv[0])


#CREATE THE TEMPLATES

from pycbc.waveform import get_td_waveform, fd_approximants, td_approximants
import pylab


def waveform(parameters, change_approx=False, delta_t=1.0/8192, f_lower=5):
    
    if change_approx == True :
        if parameters['type_merger'] == 2 or parameters['type_merger'] == 3 :
            approximant = "IMRPhenomD"
            hp, hc = get_td_waveform(#basic parameters of the integration
                                         approximant=approximant,
                                         delta_t=delta_t,
                                         f_lower=f_lower,
                                         #masses
                                         mass1=parameters['redshifted_mass1'],
                                         mass2=parameters['redshifted_mass2'],
                                         #spins
                                         spin1x=parameters['spin1x'],
                                         spin1y=parameters['spin1y'],
                                         spin1z=parameters['spin1z'],
                                         spin2x=parameters['spin2x'],
                                         spin2y=parameters['spin2y'],
                                         spin2z=parameters['spin2z'],
                                         #other parameters of the mergers
                                         #lambda1=parameters['lambda1'], 
                                         #lambda2=parameters['lambda2'],
                                         distance=parameters['dist'],
                                         inclination=parameters['inclination']
                                         )
        else :
            approximant="IMRPhenomD_NRTidalv2" #IMRPhenomPv2_NRTidalv2
            hp, hc = get_td_waveform(#basic parameters of the integration
                                         approximant=approximant,
                                         delta_t=delta_t,
                                         f_lower=f_lower,
                                         #masses
                                         mass1=parameters['redshifted_mass1'],
                                         mass2=parameters['redshifted_mass2'],
                                         #spins
                                         spin1x=parameters['spin1x'],
                                         spin1y=parameters['spin1y'],
                                         spin1z=parameters['spin1z'],
                                         spin2x=parameters['spin2x'],
                                         spin2y=parameters['spin2y'],
                                         spin2z=parameters['spin2z'],
                                         #other parameters of the mergers
                                         lambda1=parameters['lambda1'], 
                                         lambda2=parameters['lambda2'],
                                         distance=parameters['dist'],
                                         inclination=parameters['inclination']
                                         )
    elif change_approx == False :
        approximant="IMRPhenomD"
        hp, hc = get_td_waveform(#basic parameters of the integration
                                    approximant=approximant,
                                    delta_t=delta_t,
                                    f_lower=f_lower,
                                    #masses
                                    mass1=parameters['redshifted_mass1'],
                                    mass2=parameters['redshifted_mass2'],
                                    #spins
                                    #spin1x=parameters['spin1x'],
                                    #spin1y=parameters['spin1y'],
                                    #spin1z=parameters['spin1z'],
                                    #spin2x=parameters['spin2x'],
                                    #spin2y=parameters['spin2y'],
                                    #spin2z=parameters['spin2z'],
                                    #other parameters of the mergers
                                    #lambda1=parameters['lambda1'], 
                                    #lambda2=parameters['lambda2'],
                                    distance=parameters['dist'],
                                    inclination=parameters['inclination']
                                    )
        
    return hp, hc


def create_wav(alldataset = False):
    start = time.time()
    #create waveforms for every dataset included in the set of 1e9 -> 1e9 + 2048 sec
    if alldataset == False :
        parameters_file = []
        for i in range(len(parameters_conv)) :
            start_time_event_i = parameters_conv[i]['starting_time']
            time_max_event_i = parameters_conv[i]['time@Maximum']
            if 1e9 <= time_max_event_i < 1e9 + 2048 :
                hp, hc = waveform(parameters_conv[i], delta_t=1.0/8192, f_lower=7)
                parameters_conv[i]['hp'] = hp ; parameters_conv[i]['hc'] = hc
                parameters_file.append(parameters_conv[i])
        return parameters_file
    #create waveforms for every dataset of the parameters file
    elif alldataset == True :
        for i in range(len(parameters_conv)):
            hp, hc = waveform(parameters_conv[i], delta_t=1.0/8192, f_lower=7)
            parameters_conv[i]['hp'] = hp ; parameters_conv[i]['hc'] = hc
        return parameters_conv
    
    end = time.time()
    elapsed = end - start
    print('Temps d\'exécution total création template en min =', elapsed/60)

parameters_conv = create_wav()


#HOME MADE PROJECTION

import pylab
from pycbc.catalog import Merger
from pycbc.filter import resample_to_delta_t, highpass
from gwpy.timeseries import TimeSeries
from numpy import cos, sin, tensordot, sqrt, einsum


#LOOK AT PAPER MDC_ET_CE_PAPER_2012

#define radiation frame tensors
def radiation_frame_tensors(frame_parameters):
    theta, phi, psi = frame_parameters
    e_x = np.array([ -sin(phi)*sin(psi) + cos(theta)*cos(phi)*sin(psi), cos(phi)*sin(psi) + cos(theta)*sin(phi)*cos(psi), -sin(theta)*cos(psi) ])
    e_y = np.array([ -sin(phi)*cos(psi) - cos(theta)*cos(phi)*sin(psi), cos(phi)*cos(psi) - cos(theta)*sin(phi)*sin(psi), -sin(theta)*sin(psi) ])
    e_z = np.array([ sin(theta)*cos(phi), sin(theta)*sin(phi), cos(theta) ])
    return e_x, e_y, e_z

#define basis polarization tensors
def pol_tensors(frame_parameters):
    e_x, e_y, e_z = radiation_frame_tensors(frame_parameters)
    e_p = tensordot(e_x, e_x, axes=0) - tensordot(e_y, e_y, axes=0)
    e_c = tensordot(e_x, e_y, axes=0) + tensordot(e_y, e_x, axes=0)
    return e_p, e_c
 
#define interferometers tensors 
#such that the triangle with the three arms is in the xy-plane
def interf_tensors(interf_number):
    e1 = np.array([sqrt(3), -1, 0])/2
    e2 = np.array([sqrt(3), 1, 0])/2
    e3 = np.array([0, 1, 0])
    if interf_number == 1:
        d1 = ( tensordot(e1, e1, axes=0) - tensordot(e2, e2, axes=0) )/2
        return d1
    if interf_number == 2:
        d2 = ( tensordot(e2, e2, axes=0) - tensordot(e3, e3, axes=0) )/2
        return d2
    if interf_number == 3:
        d3 = ( tensordot(e3, e3, axes=0) - tensordot(e1, e1, axes=0) )/2
        return d3

#define response function and antena pattern
def antena_pattern(interf_number, frame_parameters):
    dA = interf_tensors(interf_number)
    e_p, e_c = pol_tensors(frame_parameters)
    #print('dA :', dA)
    #print('e_p :', e_p)
    #print('e_c :', e_c)
    FA_p = einsum('ij,ij',dA, e_p) ; FA_c = einsum('ij,ij',dA, e_c)
    return FA_p, FA_c
    
def response(hp, hc, interf_number, frame_parameters):
    FA_p, FA_c = antena_pattern(interf_number, frame_parameters)
    return FA_p*hp + FA_c*hc


#project and get response for all the waveforms generated and their frame_parameters
start = time.time()
for interf_number in range(1,2):
    for i in range(len(parameters_conv)):
        #get back hp and hc
        hp = parameters_conv[i]['hp'] ; hc = parameters_conv[i]['hc']
        #get back frame_parameters
        theta = parameters_conv[i]['declination'] ; phi = parameters_conv[i]['ra'] ; psi = parameters_conv[i]['polarization']
        frame_parameters = [theta, phi, psi]
        #find the response
        hA_i = response(hp, hc, interf_number, frame_parameters)
        parameters_conv[i]['strain_E'+str(interf_number)] = hA_i

end = time.time()
elapsed = end - start
print('Temps d\'exécution total projection en min =', elapsed/60)






#PROJECT THE DATA WITH THE PYCBC METHOD
import pycbc.detector as detector

list_detectors = detector.get_available_detectors()
einstein_telescope1 = list_detectors[9][0] ; einstein_telescope2 = list_detectors[10][0] ; einstein_telescope3 = list_detectors[11][0]
einstein_telescopes = [einstein_telescope1, einstein_telescope2, einstein_telescope3]
print(einstein_telescopes)

#project and get response for all the waveforms generated and their frame_parameters
start = time.time()

for telescope in ['E1'] :
    for i in range(len(parameters_conv)) :
        #get back hp and hc
        hp = parameters_conv[i]['hp'] ; hc = parameters_conv[i]['hc']
        #get back frame_parameters
        theta = parameters_conv[i]['declination'] ; phi = parameters_conv[i]['ra'] ; psi = parameters_conv[i]['polarization']
        #find the response
        detect = detector.Detector(telescope)
        h_i = detect.project_wave(hp, hc, phi, theta, psi, method='constant',reference_time=None)       
        parameters_conv[i]['strain_'+telescope+'_pycbc'] = h_i
    
end = time.time()
elapsed = end - start
print('Temps d\'exécution total projection en min =', elapsed/60)








import os
import signal

os.chdir('/Volumes/Noe')
path_dir = 'snr'
if not os.path.exists(path_dir):
    os.mkdir(path_dir)
os.chdir(path_dir)
with open('parameters_final.npy', 'wb') as file:
        #writing the file
        np.save(file, parameters_conv, allow_pickle=True, fix_imports=True)
        
        
        
pid = os.getpid()
os.kill(pid, signal.SIGKILL)

import numpy as np
import time
import os
import signal

#SNR PART

#import the data

import pycbc.frame
start = 1000000000.0
data_data = pycbc.frame.read_frame('odw_2023_main/data_files/main/E-E1_STRAIN_DATA-1000000000-2048.gwf', 'E1:STRAIN', start, start + 2048)
data_noise = pycbc.frame.read_frame('odw_2023_main/data_files/main/E-E1_STRAIN_NOISE-1000000000-2048.gwf', 'E1:STRAIN', start, start + 2048)
data_clean = data_data - data_noise

from pycbc.psd import interpolate, inverse_spectrum_truncation, welch
from pycbc import psd
from pycbc.filter import matched_filter




# VERSION INCLUDING THE LAST FEW SECONDS

def data_cutter(data, n_step, time_step, window_duration=90):
    
    if time_step == n_step:
        arg_start = int(time_step/2 * window_duration * 8192)
        arg_end = 8192*int(data.sample_times[-1])
    elif time_step % 2 == 0:
        arg_start = int(time_step/2) * window_duration * 8192
        arg_end = (int(time_step/2)+1) * window_duration * 8192
    elif time_step % 2 != 0:   
        arg_start = int(time_step/2 * window_duration * 8192)
        arg_end = int((time_step/2 + 1) * window_duration * 8192)
    data_cut = data[arg_start: arg_end]
    return data_cut



def template_matching_final(data, window_duration = 90, pycbc_ornot='homemade'):
    
    if pycbc_ornot == 'pycbc':
        name_strain = 'strain_E1_pycbc'
    elif pycbc_ornot == 'homemade':
        name_strain = 'strain_E1'
    
    start = time.time()
    
    os.chdir('/Volumes/Noe/snr')

    snrp_im1 = [(0, 0, 0, 0, 1) for i in range(34)] #len(parameters_conv)
    
    sample_rate = 8192
    
    #time_begin_snr = []
    
    n_step = 2*int(2048/window_duration)
    
    with open('parameters_final.npy', 'rb') as f_parameters_conv:
        parameters_conv = np.load(f_parameters_conv, allow_pickle=True)    
    f_parameters_conv.close()
    
    for time_step in range(n_step+1):

        start_time = time.time()

        data_cut = data_cutter(data, n_step, time_step, window_duration)
        #print(data_cut)
                
        if time_step == n_step :
            seg_len = int(len(data_cut)/8192)
        elif time_step != n_step :
            seg_len = window_duration
            
        seg_len = 4

        #psd = welch(data_cut, seg_len = seg_len, seg_stride = int(seg_len/2))
        psd = data_cut.psd(4)
        psd = interpolate(psd, data_cut.delta_f)
        psd = inverse_spectrum_truncation(psd, seg_len*sample_rate,
                                      low_frequency_cutoff=7)
        
        #pylab.figure(figsize=pylab.figaspect(0.4))
        #pylab.plot(psd.sample_frequencies, psd)
        #pylab.show()

        for i in range(34):
            
            parameters_conv_i = parameters_conv[i]
            start_time_event_i = parameters_conv_i['starting_time']
            time_max_event_i = parameters_conv_i['time@Maximum']
            number = parameters_conv_i['#']
            typemerger = parameters_conv_i['type_merger']

            #retrieving the conditionned data
            h_i = parameters_conv_i[name_strain]

            #scaling the data
            h_i.resize(len(data_cut))
            template_i = h_i.cyclic_time_shift(h_i.start_time)

            #finding the snr
            snr_i = matched_filter(template_i, data_cut, psd=psd, low_frequency_cutoff=7)
            snr_i = abs(snr_i)
            #snr_i = snr_i.crop(4 + 4, 4)

            #finding the peak
            peak_i = snr_i.numpy().argmax()
            snrp_i = snr_i[peak_i]
            
            time_i = snr_i.sample_times[peak_i]
            delta_t = abs(time_max_event_i - time_i)
            
            #keep the highest snr
            if delta_t < snrp_im1[i][4] : 
                snrp_im1[i] = (number, typemerger, snrp_i, time_i, delta_t)
                #parameters_conv_i['snr_peak'] = (snrp_i, time_i, delta_t)
                print("The time of the merger is :", time_max_event_i)
                print("We found a signal at {}s with SNR {}".format(time_i, snrp_i))
                print("Delta_t =", delta_t)
                print('')
                
            #make the snr vs time serie
            #if time_step == 0 :
                #time_begin_snr.append(snr_i.sample_times[0])
                #parameters_conv_i['snr' + str(window_duration)] = snr_i
            #else : 
                
                #retrieving the snr
                #old_snr = parameters_conv_i['snr' + str(window_duration)]
                
                #if time_step == 1:
                    #redefining the old snr as an array
                    #old_snr = np.array(old_snr)
                
                
                #keeping the first 30sec of old snr
                #left_old = 0
                #right_old = (time_step) * int(window_duration*sample_rate/2)
                #keeping_part_snr = old_snr[left_old : right_old]
                
                #keeping the last 30sec of old snr to compare
                #left_compare = time_step * int(window_duration*sample_rate/2)
                #right_compare = -1
                #comparing_part_snr = old_snr[left_compare : right_compare]
                
                #keeping the first 30sec of snr_i to compare
                #left_compare_new = 0
                #right_compare_new = len(comparing_part_snr)
                #comparing_part_new = snr_i[left_compare_new : right_compare_new]
                
                #keeping the last 30sec of new snr
                #left_new = len(comparing_part_snr)
                #right_new = -1
                #keeping_new = snr_i[left_new : right_new]
                
                #and comparing between snr's
                #new_part_snr = np.max([comparing_part_snr, comparing_part_new], axis=0)
                        
                #concatenate both snr's
                #new_snr = keeping_part_snr
                #new_snr = np.append(new_snr, new_part_snr, axis=0)
                #new_snr = np.append(new_snr, keeping_new, axis=0)
                
                #update snr
                #parameters_conv_i['snr' + str(window_duration)] = new_snr
                
                
        end_time = time.time()
        elapsed_time = end_time - start_time
        print('Temps d\'exécution step ', time_step, 'en s =', elapsed_time)
        print('')
    
                   
    end = time.time()
    elapsed = end - start
    print('Temps d\'exécution total template matching en min =', elapsed/60)
    
    return snrp_im1
        
#     #reput snr's into timeseries
#     for i in range(len(parameters_conv)):
#         snr_i = parameters_conv[i]['snr' + str(window_duration)] 
#         start_time_i = time_begin_snr[i]
#         parameters_conv[i]['snr' + str(window_duration)] = types.TimeSeries(snr_i, delta_t=1.0 / sample_rate, epoch = start_time_i)

snrp_im1 = template_matching_final(data_data, window_duration = 90, pycbc_ornot='pycbc')





with open('snr_final_data_delta_7', 'wb') as file:
        #writing the file
        np.save(file, snrp_im1, allow_pickle=True, fix_imports=True)
        
import pylab

def snr_scanning(low_freq_cutoff):
    #reading the file
    with open('snr_final_data_delta_'+str(low_freq_cutoff), 'rb') as f_snr:
        snr = np.load(f_snr, allow_pickle=True)    
    f_snr.close()
    
    #plot the evolution of the snr's with delta_t < thresh-hold
    pylab.figure(figsize=pylab.figaspect(0.4))
    pylab.xlabel('delta_t')
    pylab.ylabel('snr')
    pylab.grid()
    
    #scanning threw snr's
    count_mergers = 0
    for j in range(len(snr)):
        (number, typemerger, snrp_i, time_i, delta_t) = snr[j]
        if delta_t < 0.1:
            #plot
            label = '# :' + str(int(number)) + ', mt :' + str(int(typemerger))
            pylab.scatter(delta_t, snrp_i, label = label)
            #print
            info = {'number = ' : number, 'delta_t = ' : delta_t , 'snr_peak = ' : snrp_i}
            print(info)
            count_mergers += 1
    print('normalized number of mergers = ' + str(count_mergers/len(snr)) + ', with a thresh-hold for delta_t < 1s and snr_p > 10')
    pylab.legend()
    pylab.show()
    
    #plot the evolution of the snr's with number
    pylab.figure(figsize=pylab.figaspect(0.4))
    pylab.xlabel('number')
    pylab.ylabel('detection or not')
    pylab.grid()
    
    #scanning threw snr's
    count_mergers = 0
    for j in range(len(snr)):
        (number, typemerger, snrp_i, time_i, delta_t) = snr[j]
        #plot
        if delta_t < 1:
            yes = 1
        elif delta_t >= 1:
            yes = 0
        pylab.scatter(number, yes)
        #print
        info = {'number = ' : number , 'snr_peak = ' : snrp_i}
        print(info)
        count_mergers += 1
    print('normalized number of mergers = ' + str(count_mergers/len(snr)) + ', with a thresh-hold for delta_t < 1s and snr_p > 10')
    pylab.legend()
    pylab.show()
        
snr_scanning(low_freq_cutoff=7)

{'#': 1, 'starting_time': 999998889.639793, 'time@Maximum': 1000000937.518455, 'redshifted_mass1': 4.202228, 'redshifted_mass2': 3.818891, 'spin1': 0.0, 'spin1x': 0.0, 'spin1y': 0.0, 'spin1z': 0.0, 'spin2': 0.0, 'spin2x': 0.0, 'spin2y': 0.0, 'spin2z': 0.0, 'lambda1': 107.277557, 'lambda2': 198.823448, 'z': 1.1763, 'dist': 8295.7862, 'ra': 0.444018, 'declination': 1.132247, 'polarization': 6.185467, 'inclination': 2.802484, 'initial phase': 1.077702, 'snr_optimal': 18.054874, 'type_merger': 1}


KeyboardInterrupt: 